## 1. classical_eigensolvers

In [5]:
from algorithms.classical_eigensolvers import exact_diagonalization, lanczos_lowest_eigenvalue

hamiltonian = [
    (-1.0, "ZIX"),
    (-1.0, "IZY"),
    (0.5, "XXX"),
    (2.0, "YYX"),
    (1.5, "XYX"),
    (0.5, "XXZ"),
]

ed = exact_diagonalization(hamiltonian, n_qubits=3, k=3)
print(ed.method)
print(ed.eigenvalues)

lz = lanczos_lowest_eigenvalue(hamiltonian, n_qubits=3)
print(lz.method)
print(lz.eigenvalues)

dense-eigh
[-3.82866901 -2.97459248 -2.42526226]
dense-eigh-lowest
[-3.82866901]


## 2. phase_estimation

In [1]:
import numpy as np
from algorithms.phase_estimation import exact_phase_estimation_from_unitary

phi = 0.3
n_ancilla = 3

# |1> 是该 unitary 的 eigenstate，对应 eigenvalue exp(2πi phi)
U = np.diag([1.0, np.exp(2j * np.pi * phi)])
input_state = np.array([0.0, 1.0], dtype=np.complex128)

res = exact_phase_estimation_from_unitary(U, input_state, n_ancilla)

for bitstring, phase, prob in zip(res.bitstrings, res.phases, res.probabilities):
    if prob > 1e-3:
        print(bitstring, phase, prob)

Please first ``pip install -U qiskit`` to enable related functionality in translation module


000 0.0 0.02159321892578289
001 0.125 0.051768129535521595
010 0.25 0.5775210180698609
011 0.375 0.25933561918842774
100 0.5 0.0409067810742171
101 0.625 0.01944021679795832
110 0.75 0.014487479117612872
111 0.875 0.014947537290618543


In [3]:
import numpy as np
from backends.core import QuantumCircuit
from algorithms.phase_estimation import build_qpe_circuit, phases_from_counts

phi = 0.375
n_ancilla = 3

# U |1> = exp(2πi phi) |1>
unitary = QuantumCircuit(1)
unitary.p(0, 2 * np.pi * phi)

qpe = build_qpe_circuit(unitary, n_ancilla=n_ancilla)

print(qpe)
print(qpe.stats())

<QuantumCircuit: qubits=4, instructions=16>
{'n_qubits': 4, 'n_instructions': 16, 'gate_counts': Counter({'Phase': 7, 'H': 6, 'CPhase': 3}), 'multi_qubit_gates': 10, 'depth_approx': 13, 'parameterized': True}


In [4]:
import numpy as np
from backends._matrix_simulator import simulate_statevector, bitstring
from algorithms.phase_estimation import phases_from_counts

# QPE circuit 总 qubit 数 = 3 ancilla + 1 target = 4
# 初态 |000>|1>
initial_state = np.zeros(2**4, dtype=np.complex128)
initial_state[1] = 1.0

state = simulate_statevector(qpe, initial_state)

# 精确概率转成“伪 counts”
counts = {}
for idx, p in enumerate(np.abs(state) ** 2):
    if p > 1e-10:
        bits = bitstring(idx, qpe.n_qubits)
        ancilla_bits = bits[:n_ancilla]
        counts[ancilla_bits] = counts.get(ancilla_bits, 0.0) + float(p)

res = phases_from_counts(counts, n_ancilla)

print(counts)
print(res.bitstrings)
print(res.phases)
print(res.probabilities)

{'011': 1.0}
['011']
[0.375]
[1.]


## 3. qse

In [1]:
import numpy as np
from algorithms.qse import quantum_subspace_expansion
from algorithms.classical_eigensolvers import exact_diagonalization

def basis_state(n_qubits, index):
    state = np.zeros(2 ** n_qubits, dtype=np.complex128)
    state[index] = 1.0
    return state

# H = Z + 0.2 X
hamiltonian = [
    (1.0, "Z"),
    (0.2, "X"),
]

reference_state = basis_state(1, 0)  # |0>

# 用 X|0> = |1> 扩展子空间，因此 QSE 子空间覆盖整个 1-qubit Hilbert space
excitation_terms = [
    [(1.0, "X")]
]

qse_result = quantum_subspace_expansion(
    reference_state=reference_state,
    hamiltonian=hamiltonian,
    excitation_terms=excitation_terms,
    n_qubits=1,
)

exact_result = exact_diagonalization(hamiltonian, n_qubits=1)

print("QSE energies:", qse_result.energies)
print("Exact energies:", exact_result.eigenvalues)
print("Overlap eigenvalues:", qse_result.overlap_eigenvalues)
print("Kept dimension:", qse_result.kept_subspace_dimension)

Please first ``pip install -U qiskit`` to enable related functionality in translation module


QSE energies: [-1.0198039  1.0198039]
Exact energies: [-1.0198039  1.0198039]
Overlap eigenvalues: [1. 1.]
Kept dimension: 2


In [2]:
import numpy as np
from algorithms.qse import quantum_subspace_expansion

def basis_state(n_qubits, index):
    state = np.zeros(2 ** n_qubits, dtype=np.complex128)
    state[index] = 1.0
    return state

hamiltonian = [
    (1.0, "Z"),
    (0.2, "X"),
]

reference_state = basis_state(1, 0)

# I|0> 和 2I|0> 都和 reference state 线性相关
excitation_terms = [
    [(1.0, "I")],
    [(2.0, "I")],
    [(1.0, "X")],
]

result = quantum_subspace_expansion(
    reference_state=reference_state,
    hamiltonian=hamiltonian,
    excitation_terms=excitation_terms,
    n_qubits=1,
    regularization=1e-8,
)

print("QSE energies:", result.energies)
print("Overlap eigenvalues:", result.overlap_eigenvalues)
print("Kept dimension:", result.kept_subspace_dimension)

QSE energies: [-1.0198039  1.0198039]
Overlap eigenvalues: [-4.53155957e-16  9.06674729e-18  1.00000000e+00  3.00000000e+00]
Kept dimension: 2


In [3]:
import numpy as np
from algorithms.qse import quantum_subspace_expansion, singles_doubles_qse_pool

def basis_state_from_bitstring(bitstring):
    n_qubits = len(bitstring)
    index = int(bitstring, 2)
    state = np.zeros(2 ** n_qubits, dtype=np.complex128)
    state[index] = 1.0
    return state

n_qubits = 4
n_electrons = 2

# Hartree-Fock-like |1100>
reference_state = basis_state_from_bitstring("1100")

hamiltonian = [
    (0.1, "ZIII"),
    (-0.2, "IZII"),
    (0.3, "IIZI"),
    (0.4, "IIIZ"),
    (0.05, "XXII"),
]

pool = singles_doubles_qse_pool(n_qubits, n_electrons)

result = quantum_subspace_expansion(
    reference_state=reference_state,
    hamiltonian=hamiltonian,
    excitation_terms=pool,
    n_qubits=n_qubits,
)

print("Lowest QSE energies:", result.energies[:5])
print("Subspace dimension:", result.kept_subspace_dimension)
print("First pool operator:", pool[0])

Lowest QSE energies: [-0.8        -0.40413813 -0.20413813  0.20413813  0.40413813]
Subspace dimension: 6
First pool operator: [(0.5, 'XIXI'), (0.5, 'YIYI')]


## 4. quantum_chemistry

In [1]:
import numpy as np

from algorithms.quantum_chemistry import (
    mp2_energy_from_integrals,
    fci_reference_energy,
    chemical_accuracy_error,
    state_fidelity,
    natural_orbital_occupations,
    particle_number_expectation,
    spin_z_expectation,
    active_space_indices,
    freeze_core_energy_shift,
    symmetry_project_state,
    commutator_norm,
)

# 1. FCI reference energy，使用库标准 Hamiltonian 格式
hamiltonian = [
    (1.0, "Z"),
    (0.2, "X"),
]

e_fci = fci_reference_energy(hamiltonian, n_qubits=1)
print("FCI energy:", e_fci)

err = chemical_accuracy_error(estimated_energy=-1.018, reference_energy=e_fci)
print(err)


# 2. MP2 reference energy
eps = np.array([0.0, 1.0])
eri = np.zeros((2, 2, 2, 2))

# chemist-order: eri[p,q,r,s] = (pq|rs)
eri[0, 1, 0, 1] = 0.2  # (0,1|0,1)

mp2 = mp2_energy_from_integrals(
    eps=eps,
    eri_mo=eri,
    n_electrons=1,
    hf_energy=-1.0,
)

print(mp2)
# MP2Result(correlation_energy=-0.02, total_energy=-1.02, ...)


# 3. Fidelity
state_a = np.array([1.0, 0.0], dtype=np.complex128)
state_b = np.array([1.0j, 0.0], dtype=np.complex128)

fid = state_fidelity(state_a, state_b)
print("Fidelity:", fid.fidelity)
print("Phase overlap:", fid.phase)


# 4. Natural orbital occupations
one_rdm = np.array([
    [1.8, 0.1],
    [0.1, 0.2],
], dtype=np.complex128)

occ = natural_orbital_occupations(one_rdm)
print("Natural occupations:", occ)


# 5. 粒子数和 Sz
state = np.zeros(4, dtype=np.complex128)
state[3] = 2.0  # 非归一化 |11>

print("N:", particle_number_expectation(state, n_qubits=2))
print("Sz:", spin_z_expectation(state, n_qubits=2))


# 6. 对称性投影
bell = np.array([1, 0, 0, 1], dtype=np.complex128) / np.sqrt(2)
projected = symmetry_project_state(
    bell,
    n_qubits=2,
    n_particles=2,
    sz=0.0,
)

print("Projected state:", projected)


# 7. Active space
active = active_space_indices(
    n_orbitals=10,
    n_core=2,
    n_active=4,
    n_virtual=1,
)
print("Active orbitals:", active)


# 8. Frozen-core scalar shift
shift = freeze_core_energy_shift(
    core_orbital_energies=[-20.0, -1.0],
    nuclear_repulsion=10.0,
    core_coulomb_exchange_shift=3.0,
)
print("Frozen-core shift:", shift)


# 9. Commutator norm
X = np.array([[0, 1], [1, 0]], dtype=np.complex128)
Z = np.array([[1, 0], [0, -1]], dtype=np.complex128)

print("||[X,Z]||:", commutator_norm(X, Z))

Please first ``pip install -U qiskit`` to enable related functionality in translation module


FCI energy: -1.019803902718557
{'hartree_error': 0.0018039027185570156, 'absolute_hartree_error': 0.0018039027185570156, 'kcal_per_mol_error': 1.1319660460688827, 'absolute_kcal_per_mol_error': 1.1319660460688827, 'within_chemical_accuracy': False}
MP2Result(correlation_energy=-0.020000000000000004, total_energy=-1.02, amplitudes_shape=(1, 1, 1, 1), skipped_denominators=0, convention='closed_shell_chemist')
Fidelity: 1.0
Phase overlap: 1j
Natural occupations: [1.80622577 0.19377423]
N: 2.0
Sz: 0.0
Projected state: [0.+0.j 0.+0.j 0.+0.j 1.+0.j]
Active orbitals: [2, 3, 4, 5]
Frozen-core shift: -29.0
||[X,Z]||: 2.8284271247461903


## 5. trotter

In [1]:
from algorithms.trotter import (
    first_order_trotter_circuit,
    second_order_trotter_circuit,
    trotter_info,
)

hamiltonian = [
    (-1.0, "ZI"),
    (-1.0, "IZ"),
    (0.5, "XX"),
]

time = 0.3
n_steps = 4
n_qubits = 2

qc1 = first_order_trotter_circuit(
    hamiltonian,
    time=time,
    n_steps=n_steps,
    n_qubits=n_qubits,
)

qc2 = second_order_trotter_circuit(
    hamiltonian,
    time=time,
    n_steps=n_steps,
    n_qubits=n_qubits,
)

print(qc1)
print(qc2)

print(trotter_info(hamiltonian, time=time, n_steps=n_steps, order=1))
print(trotter_info(hamiltonian, time=time, n_steps=n_steps, order=2))

Please first ``pip install -U qiskit`` to enable related functionality in translation module


<QuantumCircuit: qubits=2, instructions=36>
<QuantumCircuit: qubits=2, instructions=72>
TrotterStepInfo(order=1, time=0.3, n_steps=4, n_terms=3, gate_count_estimate=36)
TrotterStepInfo(order=2, time=0.3, n_steps=4, n_terms=3, gate_count_estimate=72)


In [2]:
import numpy as np

from algorithms.trotter import second_order_trotter_circuit
from backends._matrix_simulator import simulate_statevector

hamiltonian = [
    (1.0, "X"),
    (0.7, "Z"),
]

qc = second_order_trotter_circuit(
    hamiltonian,
    time=0.8,
    n_steps=8,
    n_qubits=1,
)

initial_state = np.array([1.0, 0.0], dtype=np.complex128)
final_state = simulate_statevector(qc, initial_state)

print(final_state)

[ 5.60349564e-01-0.4760265j  -1.11022302e-16-0.67779579j]


In [3]:
from algorithms.trotter import trotter_circuit

hamiltonian = [
    (-1.0, "ZI"),
    (-1.0, "IZ"),
    (0.5, "XX"),
]

qc = trotter_circuit(
    hamiltonian,
    time=1.0,
    n_steps=10,
    order=2,
    n_qubits=2,
)

print(qc)

<QuantumCircuit: qubits=2, instructions=180>


## 6. vqd

In [1]:
import numpy as np

from algorithms.vqd import (
    deflated_eigensystem,
    vqd_objective_from_matrix,
    state_overlap,
)

# 一个简单三能级 Hamiltonian
H = np.diag([0.0, 1.0, 2.0]).astype(np.complex128)

# 已经找到的基态 |0>
ground_state = np.array([1.0, 0.0, 0.0], dtype=np.complex128)

# beta 要大于目标能级间隔，才能把基态推高
betas = [5.0]

vals, vecs = deflated_eigensystem(
    H,
    previous_states=[ground_state],
    betas=betas,
)

print("Deflated eigenvalues:", vals)
print("Lowest deflated state:", vecs[:, 0])

# 检查最低 deflated state 是否正交于基态
print("Overlap with ground:", state_overlap(vecs[:, 0], ground_state))

Please first ``pip install -U qiskit`` to enable related functionality in translation module


Deflated eigenvalues: [1. 2. 5.]
Lowest deflated state: [0.+0.j 1.+0.j 0.+0.j]
Overlap with ground: 0.0


In [3]:
import numpy as np
from algorithms.vqd import vqd_objective_from_matrix

H = np.diag([0.0, 1.0, 2.0]).astype(np.complex128)

ground_state = np.array([1.0, 0.0, 0.0], dtype=np.complex128)
candidate_state = np.array([0.0, 1.0, 0.0], dtype=np.complex128)

result = vqd_objective_from_matrix(
    H,
    candidate_state=candidate_state,
    previous_states=[ground_state],
    betas=[5.0],
)

print("Energy:", result.energy)
print("Overlaps:", result.overlaps)
print("Penalties:", result.penalties)
print("Total:", result.total)

Energy: 1.0
Overlaps: [0.0]
Penalties: [0.0]
Total: 1.0


In [4]:
import numpy as np
from algorithms.vqd import gram_schmidt_states

states = [
    np.array([1.0, 0.0], dtype=np.complex128),
    np.array([2.0, 0.0], dtype=np.complex128),  # 和第一个线性相关，会被丢弃
    np.array([1.0, 1.0], dtype=np.complex128),
]

basis = gram_schmidt_states(states)

for v in basis:
    print(v)

[1.+0.j 0.+0.j]
[0.+0.j 1.+0.j]


## 7. shadow

In [1]:
import numpy as np

from backends.core import QuantumCircuit
from backends._matrix_simulator import simulate_statevector, sample_counts
from algorithms.shadow import ClassicalShadow


class NumpySamplingBackend:
    def __init__(self, seed=123):
        self.rng = np.random.default_rng(seed)

    def run_sampling(self, circuit, shots, measure_qubits):
        state = simulate_statevector(circuit)
        return sample_counts(
            state,
            shots=shots,
            measure_qubits=measure_qubits,
            seed=int(self.rng.integers(2**32)),
        )


# 制备 |+i> = S H |0>
qc = QuantumCircuit(1)
qc.h(0)
qc.s(0)

backend = NumpySamplingBackend(seed=1)

shadow = ClassicalShadow(
    backend=backend,
    circuit=qc,
    seed=2,
)

shadow.collect(5000)

print("<X> =", shadow.estimate_observable("X"))
print("<Y> =", shadow.estimate_observable("Y"))
print("<Z> =", shadow.estimate_observable("Z"))
print("<I> =", shadow.estimate_observable("I"))

Please first ``pip install -U qiskit`` to enable related functionality in translation module


<X> = 0.0054
<Y> = 0.996
<Z> = -0.009
<I> = 1.0


In [2]:
hamiltonian = [
    (1.0, "I"),
    (0.5, "Y"),
]

energy = shadow.estimate_hamiltonian(hamiltonian)
print("Estimated energy:", energy)

Estimated energy: (1.498+0j)


In [3]:
from backends.core import QuantumCircuit
from algorithms.shadow import ClassicalShadow

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

shadow = ClassicalShadow(
    backend=NumpySamplingBackend(seed=3),
    circuit=qc,
    seed=4,
)

shadow.collect(10000)

print("<XX> =", shadow.estimate_observable("XX"))
print("<YY> =", shadow.estimate_observable("YY"))
print("<ZZ> =", shadow.estimate_observable("ZZ"))
print("<ZI> =", shadow.estimate_observable("ZI"))
print("<IZ> =", shadow.estimate_observable("IZ"))
print("<II> =", shadow.estimate_observable("II"))

<XX> = 0.9828
<YY> = -0.9855
<ZZ> = 0.9783
<ZI> = 0.0321
<IZ> = 0.024
<II> = 1.0


In [4]:
rho_hat = shadow.estimate_density_matrix()
print(rho_hat)

[[ 0.5086  +0.j       -0.002475+0.007425j -0.0069  -0.0024j
   0.492075+0.009225j]
 [-0.002475-0.007425j  0.00745 +0.j       -0.000675-0.008775j
   0.0012  +0.01155j ]
 [-0.0069  +0.0024j   -0.000675+0.008775j  0.0034  +0.j
   0.011025-0.004275j]
 [ 0.492075-0.009225j  0.0012  -0.01155j   0.011025+0.004275j
   0.48055 +0.j      ]]
